# 第26章　領域を広げる ― びまん性肺疾患・大腸・甲状腺・心臓CT**『本格実装 医療診断支援AI（実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-impl

## 領域を広げる② ― CTコロノグラフィとポリープ検出

In [ ]:
def is_mobile(supine_lesion, prone_lesion, wall_frame, move_thr_mm=10):    # 便は体位で動く／ポリープは壁に固着して動かない    d = registered_distance(supine_lesion, prone_lesion, wall_frame)    return {"moved_mm": round(d, 1), "likely_stool": d > move_thr_mm}

## 領域を広げる③ ― 甲状腺超音波とTI-RADS

In [ ]:
def taller_than_wide(nodule_mask, spacing_mm, plane):    """spacing_mm=(縦, 横) の画素間隔[mm]。画素数の比では実寸の比にならない。    plane は断面の種別（"transverse" 等）。縦横の向きが決まらなければ判定しない。"""    if plane not in ("transverse",) or nodule_mask.sum() == 0:        return {"state": "評価不能（断面種別が不明、またはマスクが空）"}    import math, numbers    spacing_mm = np.asarray(spacing_mm, dtype=object)    if (spacing_mm.shape != (2,) or not all(            isinstance(v, numbers.Real) and not isinstance(v, bool)            and math.isfinite(v) and v > 0 for v in spacing_mm)):        return {"state": "評価不能（画素間隔が数値でない、非有限、0以下、または2要素でない）"}    ys, xs = np.where(nodule_mask)    ap = (ys.max() - ys.min() + 1) * spacing_mm[0]   # 前後径[mm]（画像の縦）    tr = (xs.max() - xs.min() + 1) * spacing_mm[1]   # 横径[mm]    if tr <= 0:        return {"state": "評価不能（横径が0）"}    return {"ap_mm": round(ap, 1), "tr_mm": round(tr, 1),            "ap_tr_ratio": round(ap / tr, 2),            "taller_than_wide": ap > tr, "state": "ok"}   # 悪性を示唆する形状

## もう一つの定量 ― 心臓CTと冠動脈石灰化スコア

In [ ]:
def agatston_lesion(area_mm2, max_hu):    w = 1 if max_hu < 200 else 2 if max_hu < 300 else 3 if max_hu < 400 else 4    return area_mm2 * w                        # スライス内の病変ごとに算出し、全スライス・全冠動脈で合計# 合計スコアを 0 / 1-99 / 100-399 / 400+ の帯に落としてリスク層別する